# End-to-End Safety Validation System for Menopause Education AI

## Overview
This notebook implements a comprehensive safety validation layer for an AI system focused on menopause education. The system ensures that all interactions are safe, accurate, medically sound, and free from harmful biases.

## Architecture Components
1. **Input Validation Layer** - Validates and sanitizes user inputs
2. **Content Safety Filter** - Screens for inappropriate content
3. **Medical Accuracy Validator** - Ensures medical information accuracy
4. **Bias Detection System** - Identifies and mitigates potential biases
5. **Output Safety Monitor** - Final validation before response delivery
6. **Audit & Logging System** - Comprehensive logging for compliance

## Safety Standards
- HIPAA compliance for health information
- Medical device software standards (IEC 62304)
- AI ethics guidelines for healthcare
- Cultural sensitivity requirements

In [ ]:
# Import required libraries
import re
import json
import logging
import hashlib
import datetime
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
from enum import Enum
import warnings
from collections import defaultdict
import uuid

# Scientific computing
import numpy as np
import pandas as pd

# NLP libraries
from textblob import TextBlob
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('vader_lexicon', quiet=True)

## 1. Core Safety Classes and Enums

We define the foundational classes that will be used throughout the safety validation system.

In [ ]:
class SafetyLevel(Enum):
    """Safety levels for content classification"""
    SAFE = "safe"
    CAUTION = "caution"
    WARNING = "warning"
    BLOCKED = "blocked"

class ValidationResult(Enum):
    """Results of validation checks"""
    PASS = "pass"
    FAIL = "fail"
    REVIEW = "review"

@dataclass
class SafetyCheckResult:
    """Result of a safety validation check"""
    check_name: str
    result: ValidationResult
    safety_level: SafetyLevel
    confidence_score: float
    details: Dict[str, Any]
    timestamp: datetime.datetime
    recommendations: List[str]

@dataclass
class UserInteraction:
    """Structure for user interaction data"""
    session_id: str
    user_input: str
    user_demographics: Dict[str, Any]
    context: Dict[str, Any]
    timestamp: datetime.datetime
    interaction_id: str

## 2. Input Validation Layer

The first line of defense - validates and sanitizes all incoming user inputs to ensure they meet safety requirements.

In [ ]:
class InputValidator:
    """Validates and sanitizes user inputs"""
    
    def __init__(self):
        self.max_input_length = 2000
        self.blocked_patterns = [
            r'\b(?:suicide|self\s*harm|kill\s*myself)\b',
            r'\b(?:drug\s*names?|medication\s*names?)\b',
            r'\b(?:diagnosis|diagnose\s*me)\b',
            r'\b(?:emergency|urgent|immediate\s*help)\b'
        ]
        self.suspicious_patterns = [
            r'\b(?:personal\s*info|ssn|social\s*security)\b',
            r'\b(?:credit\s*card|bank\s*account)\b',
            r'\b(?:password|login\s*details)\b'
        ]
        
    def validate_input(self, user_input: str, context: Dict[str, Any]) -> SafetyCheckResult:
        """Comprehensive input validation"""
        
        details = {}
        recommendations = []
        safety_level = SafetyLevel.SAFE
        result = ValidationResult.PASS
        confidence_score = 1.0
        
        # Length validation
        if len(user_input) > self.max_input_length:
            safety_level = SafetyLevel.BLOCKED
            result = ValidationResult.FAIL
            details['length_violation'] = f"Input too long: {len(user_input)} characters"
            recommendations.append("Please shorten your question")
            
        # Check for blocked content
        for pattern in self.blocked_patterns:
            if re.search(pattern, user_input, re.IGNORECASE):
                safety_level = SafetyLevel.BLOCKED
                result = ValidationResult.FAIL
                details['blocked_content'] = f"Contains blocked pattern: {pattern}"
                recommendations.append("This type of question requires professional medical consultation")
                confidence_score = 0.9
                break
                
        # Check for suspicious content
        for pattern in self.suspicious_patterns:
            if re.search(pattern, user_input, re.IGNORECASE):
                safety_level = SafetyLevel.WARNING
                result = ValidationResult.REVIEW
                details['suspicious_content'] = f"Contains suspicious pattern: {pattern}"
                recommendations.append("Please avoid sharing personal information")
                confidence_score = 0.8
                
        # Sanitize input
        sanitized_input = self._sanitize_input(user_input)
        details['sanitized_input'] = sanitized_input
        details['original_length'] = len(user_input)
        details['sanitized_length'] = len(sanitized_input)
        
        return SafetyCheckResult(
            check_name="input_validation",
            result=result,
            safety_level=safety_level,
            confidence_score=confidence_score,
            details=details,
            timestamp=datetime.datetime.now(),
            recommendations=recommendations
        )
    
    def _sanitize_input(self, text: str) -> str:
        """Sanitize user input"""
        # Remove potentially harmful characters
        sanitized = re.sub(r'[<>"\'\\/]', '', text)
        # Normalize whitespace
        sanitized = re.sub(r'\s+', ' ', sanitized).strip()
        return sanitized

## 3. Content Safety Filter

Advanced content filtering system that screens for inappropriate, harmful, or off-topic content using multiple detection methods.

In [ ]:
class ContentSafetyFilter:
    """Advanced content safety filtering system"""
    
    def __init__(self):
        self.menopause_keywords = [
            'menopause', 'perimenopause', 'postmenopause', 'hot flashes', 'hot flushes',
            'hormone therapy', 'estrogen', 'progesterone', 'vaginal dryness',
            'mood swings', 'night sweats', 'irregular periods', 'bone health',
            'osteoporosis', 'cardiovascular health', 'weight gain', 'sleep issues'
        ]
        
        self.harmful_topics = [
            'self-harm', 'suicide', 'violence', 'illegal drugs', 'extremism',
            'hate speech', 'discrimination', 'harassment'
        ]
        
        self.medical_disclaimers = [
            'diagnosis', 'treatment', 'prescription', 'dosage', 'medication'
        ]
    
    def filter_content(self, content: str, context: Dict[str, Any]) -> SafetyCheckResult:
        """Comprehensive content safety filtering"""
        
        details = {}
        recommendations = []
        safety_level = SafetyLevel.SAFE
        result = ValidationResult.PASS
        confidence_score = 1.0
        
        # Topic relevance check
        relevance_score = self._check_topic_relevance(content)
        details['topic_relevance_score'] = relevance_score
        
        if relevance_score < 0.3:
            safety_level = SafetyLevel.CAUTION
            result = ValidationResult.REVIEW
            recommendations.append("Question may be outside menopause education scope")
            confidence_score = 0.7
        
        # Harmful content detection
        harmful_score = self._detect_harmful_content(content)
        details['harmful_content_score'] = harmful_score
        
        if harmful_score > 0.5:
            safety_level = SafetyLevel.BLOCKED
            result = ValidationResult.FAIL
            recommendations.append("Content contains potentially harmful elements")
            confidence_score = 0.9
        
        # Medical disclaimer requirement check
        needs_disclaimer = self._check_medical_disclaimer_needed(content)
        details['needs_medical_disclaimer'] = needs_disclaimer
        
        if needs_disclaimer:
            recommendations.append("Response should include medical disclaimer")
        
        # Sentiment analysis
        sentiment = self._analyze_sentiment(content)
        details['sentiment'] = sentiment
        
        if sentiment['compound'] < -0.5:
            safety_level = SafetyLevel.CAUTION
            recommendations.append("User may be experiencing distress - provide supportive response")
        
        return SafetyCheckResult(
            check_name="content_safety_filter",
            result=result,
            safety_level=safety_level,
            confidence_score=confidence_score,
            details=details,
            timestamp=datetime.datetime.now(),
            recommendations=recommendations
        )
    
    def _check_topic_relevance(self, content: str) -> float:
        """Check if content is relevant to menopause education"""
        content_lower = content.lower()
        matches = sum(1 for keyword in self.menopause_keywords if keyword in content_lower)
        return min(1.0, matches / 3)  # Normalize to 0-1 range
    
    def _detect_harmful_content(self, content: str) -> float:
        """Detect potentially harmful content"""
        content_lower = content.lower()
        matches = sum(1 for topic in self.harmful_topics if topic in content_lower)
        return min(1.0, matches / 2)  # Normalize to 0-1 range
    
    def _check_medical_disclaimer_needed(self, content: str) -> bool:
        """Check if medical disclaimer is needed"""
        content_lower = content.lower()
        return any(term in content_lower for term in self.medical_disclaimers)
    
    def _analyze_sentiment(self, content: str) -> Dict[str, float]:
        """Analyze sentiment of content"""
        blob = TextBlob(content)
        return {
            'polarity': blob.sentiment.polarity,
            'subjectivity': blob.sentiment.subjectivity,
            'compound': blob.sentiment.polarity  # Simplified compound score
        }

## 4. Medical Accuracy Validator

Ensures that medical information provided is accurate and appropriate for the educational context.

In [ ]:
class MedicalAccuracyValidator:
    """Validates medical accuracy of content"""
    
    def __init__(self):
        # Medical fact database (simplified for demo)
        self.medical_facts = {
            'menopause_age_range': (45, 55),
            'perimenopause_duration': (2, 10),  # years
            'hormone_therapy_risks': ['blood clots', 'stroke', 'breast cancer'],
            'common_symptoms': [
                'hot flashes', 'night sweats', 'mood changes', 'sleep problems',
                'vaginal dryness', 'decreased libido', 'weight gain'
            ]
        }
        
        self.contraindicated_advice = [
            'stop taking prescribed medication',
            'self-diagnose',
            'ignore doctor advice',
            'use unproven treatments'
        ]
        
        self.red_flag_statements = [
            r'\b(?:definitely|certainly|guaranteed)\s+(?:cure|fix|eliminate)\b',
            r'\b(?:never|always)\s+(?:happens|occurs|causes)\b',
            r'\b(?:all|every)\s+(?:women|people)\s+(?:experience|have)\b'
        ]
    
    def validate_medical_content(self, content: str, context: Dict[str, Any]) -> SafetyCheckResult:
        """Validate medical accuracy of content"""
        
        details = {}
        recommendations = []
        safety_level = SafetyLevel.SAFE
        result = ValidationResult.PASS
        confidence_score = 1.0
        
        # Check for contraindicated advice
        contraindication_score = self._check_contraindicated_advice(content)
        details['contraindication_score'] = contraindication_score
        
        if contraindication_score > 0.5:
            safety_level = SafetyLevel.BLOCKED
            result = ValidationResult.FAIL
            recommendations.append("Content contains potentially harmful medical advice")
            confidence_score = 0.9
        
        # Check for red flag statements (overly definitive claims)
        red_flags = self._detect_red_flag_statements(content)
        details['red_flags'] = red_flags
        
        if red_flags:
            safety_level = SafetyLevel.WARNING
            result = ValidationResult.REVIEW
            recommendations.append("Avoid overly definitive medical statements")
            confidence_score = 0.7
        
        # Fact consistency check
        fact_consistency = self._check_fact_consistency(content)
        details['fact_consistency'] = fact_consistency
        
        if fact_consistency < 0.8:
            safety_level = SafetyLevel.CAUTION
            result = ValidationResult.REVIEW
            recommendations.append("Please verify medical facts against current guidelines")
            confidence_score = 0.6
        
        # Evidence level assessment
        evidence_level = self._assess_evidence_level(content)
        details['evidence_level'] = evidence_level
        
        if evidence_level == 'low':
            recommendations.append("Include evidence level disclaimer")
        
        return SafetyCheckResult(
            check_name="medical_accuracy_validation",
            result=result,
            safety_level=safety_level,
            confidence_score=confidence_score,
            details=details,
            timestamp=datetime.datetime.now(),
            recommendations=recommendations
        )
    
    def _check_contraindicated_advice(self, content: str) -> float:
        """Check for potentially harmful medical advice"""
        content_lower = content.lower()
        matches = sum(1 for advice in self.contraindicated_advice if advice in content_lower)
        return min(1.0, matches / 2)
    
    def _detect_red_flag_statements(self, content: str) -> List[str]:
        """Detect overly definitive medical statements"""
        red_flags = []
        for pattern in self.red_flag_statements:
            matches = re.findall(pattern, content, re.IGNORECASE)
            red_flags.extend(matches)
        return red_flags
    
    def _check_fact_consistency(self, content: str) -> float:
        """Check consistency with known medical facts"""
        # Simplified fact checking - in production, this would use
        # a comprehensive medical knowledge base
        score = 0.9  # Default high score
        
        # Check age ranges mentioned
        age_matches = re.findall(r'\b(\d{2})\s*(?:years?\s+old|age)', content)
        for age_str in age_matches:
            age = int(age_str)
            if 'menopause' in content.lower() and not (40 <= age <= 60):
                score -= 0.2
        
        return max(0.0, score)
    
    def _assess_evidence_level(self, content: str) -> str:
        """Assess the evidence level of medical claims"""
        evidence_keywords = {
            'high': ['clinical trial', 'randomized', 'meta-analysis', 'systematic review'],
            'medium': ['observational study', 'cohort study', 'case-control'],
            'low': ['case report', 'expert opinion', 'anecdotal']
        }
        
        content_lower = content.lower()
        
        for level, keywords in evidence_keywords.items():
            if any(keyword in content_lower for keyword in keywords):
                return level
        
        return 'unknown'

## 5. Bias Detection System

Detects and mitigates various forms of bias including cultural, age, socioeconomic, and accessibility biases.

In [ ]:
class BiasDetectionSystem:
    """Detects and mitigates various forms of bias"""
    
    def __init__(self):
        self.bias_patterns = {
            'cultural': [
                r'\b(?:all|most|typical)\s+(?:women|cultures|societies)\b',
                r'\bnormal\s+(?:for|in)\s+(?:western|eastern|american|european)\b'
            ],
            'socioeconomic': [
                r'\b(?:expensive|costly|premium)\s+(?:treatments?|options?)\s+(?:are\s+)?(?:better|best)\b',
                r'\b(?:just|simply)\s+(?:buy|purchase|get)\b'
            ],
            'age': [
                r'\b(?:too\s+old|past\s+prime|over\s+the\s+hill)\b',
                r'\b(?:young|younger)\s+women\s+(?:don\'t|won\'t|can\'t)\b'
            ],
            'accessibility': [
                r'\b(?:just|simply|easily)\s+(?:exercise|walk|run)\b',
                r'\b(?:everyone|all\s+women)\s+(?:can|should|must)\b'
            ]
        }
        
        self.inclusive_language_check = {
            'gender_inclusive': ['people who menstruate', 'individuals', 'those experiencing'],
            'ability_inclusive': ['when possible', 'if able', 'as appropriate'],
            'cultural_inclusive': ['in some cultures', 'for many people', 'commonly']
        }
    
    def detect_bias(self, content: str, context: Dict[str, Any]) -> SafetyCheckResult:
        """Comprehensive bias detection"""
        
        details = {}
        recommendations = []
        safety_level = SafetyLevel.SAFE
        result = ValidationResult.PASS
        confidence_score = 1.0
        
        # Detect different types of bias
        bias_scores = {}
        total_bias_score = 0
        
        for bias_type, patterns in self.bias_patterns.items():
            score = self._calculate_bias_score(content, patterns)
            bias_scores[bias_type] = score
            total_bias_score += score
            
            if score > 0.3:
                recommendations.append(f"Review content for {bias_type} bias")
        
        details['bias_scores'] = bias_scores
        details['total_bias_score'] = total_bias_score
        
        # Overall bias assessment
        if total_bias_score > 0.8:
            safety_level = SafetyLevel.WARNING
            result = ValidationResult.REVIEW
            confidence_score = 0.7
            recommendations.append("High bias detected - content needs revision")
        elif total_bias_score > 0.5:
            safety_level = SafetyLevel.CAUTION
            result = ValidationResult.REVIEW
            confidence_score = 0.8
            recommendations.append("Moderate bias detected - consider revisions")
        
        # Check for inclusive language
        inclusivity_score = self._check_inclusive_language(content)
        details['inclusivity_score'] = inclusivity_score
        
        if inclusivity_score < 0.5:
            recommendations.append("Consider using more inclusive language")
        
        # Cultural sensitivity check
        cultural_sensitivity = self._check_cultural_sensitivity(content)
        details['cultural_sensitivity'] = cultural_sensitivity
        
        if cultural_sensitivity < 0.6:
            recommendations.append("Review content for cultural sensitivity")
        
        return SafetyCheckResult(
            check_name="bias_detection",
            result=result,
            safety_level=safety_level,
            confidence_score=confidence_score,
            details=details,
            timestamp=datetime.datetime.now(),
            recommendations=recommendations
        )
    
    def _calculate_bias_score(self, content: str, patterns: List[str]) -> float:
        """Calculate bias score for specific bias type"""
        matches = 0
        for pattern in patterns:
            if re.search(pattern, content, re.IGNORECASE):
                matches += 1
        return min(1.0, matches / len(patterns))
    
    def _check_inclusive_language(self, content: str) -> float:
        """Check for inclusive language usage"""
        content_lower = content.lower()
        inclusive_count = 0
        total_phrases = 0
        
        for category, phrases in self.inclusive_language_check.items():
            total_phrases += len(phrases)
            for phrase in phrases:
                if phrase in content_lower:
                    inclusive_count += 1
        
        return inclusive_count / max(1, total_phrases) if total_phrases > 0 else 0.5
    
    def _check_cultural_sensitivity(self, content: str) -> float:
        """Check cultural sensitivity of content"""
        # Simplified cultural sensitivity check
        sensitive_terms = ['traditional', 'cultural', 'diverse', 'varies', 'individual']
        insensitive_terms = ['primitive', 'backward', 'weird', 'strange', 'abnormal']
        
        content_lower = content.lower()
        sensitive_count = sum(1 for term in sensitive_terms if term in content_lower)
        insensitive_count = sum(1 for term in insensitive_terms if term in content_lower)
        
        # Score based on balance of sensitive vs insensitive language
        total_terms = sensitive_count + insensitive_count
        if total_terms == 0:
            return 0.7  # Neutral score
        
        return sensitive_count / total_terms

## 6. Output Safety Monitor

Final validation layer that monitors AI-generated responses before delivery to users.

In [ ]:
class OutputSafetyMonitor:
    """Final safety validation for AI outputs"""
    
    def __init__(self):
        self.required_disclaimers = {
            'medical': "This information is for educational purposes only and should not replace professional medical advice.",
            'individual_variation': "Individual experiences may vary. Consult with a healthcare provider for personalized guidance.",
            'emergency': "If you are experiencing a medical emergency, please contact emergency services immediately."
        }
        
        self.quality_metrics = {
            'min_length': 50,
            'max_length': 2000,
            'min_readability_score': 60,
            'min_completeness_score': 0.7
        }
    
    def monitor_output(self, output: str, context: Dict[str, Any]) -> SafetyCheckResult:
        """Comprehensive output safety monitoring"""
        
        details = {}
        recommendations = []
        safety_level = SafetyLevel.SAFE
        result = ValidationResult.PASS
        confidence_score = 1.0
        
        # Quality checks
        quality_score = self._assess_output_quality(output)
        details['quality_score'] = quality_score
        
        if quality_score < 0.6:
            safety_level = SafetyLevel.CAUTION
            result = ValidationResult.REVIEW
            recommendations.append("Output quality below threshold")
            confidence_score = 0.7
        
        # Disclaimer compliance check
        disclaimer_compliance = self._check_disclaimer_compliance(output, context)
        details['disclaimer_compliance'] = disclaimer_compliance
        
        if not disclaimer_compliance['compliant']:
            safety_level = SafetyLevel.WARNING
            result = ValidationResult.REVIEW
            recommendations.extend(disclaimer_compliance['missing_disclaimers'])
            confidence_score = 0.8
        
        # Completeness check
        completeness_score = self._assess_completeness(output, context)
        details['completeness_score'] = completeness_score
        
        if completeness_score < self.quality_metrics['min_completeness_score']:
            recommendations.append("Response may be incomplete")
        
        # Readability assessment
        readability_score = self._assess_readability(output)
        details['readability_score'] = readability_score
        
        if readability_score < self.quality_metrics['min_readability_score']:
            recommendations.append("Consider simplifying language for better readability")
        
        # Safety signal detection
        safety_signals = self._detect_safety_signals(output)
        details['safety_signals'] = safety_signals
        
        if safety_signals['critical_signals']:
            safety_level = SafetyLevel.BLOCKED
            result = ValidationResult.FAIL
            recommendations.append("Critical safety signals detected")
            confidence_score = 0.9
        
        return SafetyCheckResult(
            check_name="output_safety_monitor",
            result=result,
            safety_level=safety_level,
            confidence_score=confidence_score,
            details=details,
            timestamp=datetime.datetime.now(),
            recommendations=recommendations
        )
    
    def _assess_output_quality(self, output: str) -> float:
        """Assess overall output quality"""
        score = 1.0
        
        # Length check
        if len(output) < self.quality_metrics['min_length']:
            score -= 0.3
        elif len(output) > self.quality_metrics['max_length']:
            score -= 0.2
        
        # Structure check (has sentences)
        sentences = sent_tokenize(output)
        if len(sentences) < 2:
            score -= 0.2
        
        # Basic grammar check (simplified)
        if not output.strip().endswith(('.', '!', '?')):
            score -= 0.1
        
        return max(0.0, score)
    
    def _check_disclaimer_compliance(self, output: str, context: Dict[str, Any]) -> Dict[str, Any]:
        """Check if appropriate disclaimers are included"""
        output_lower = output.lower()
        missing_disclaimers = []
        
        # Check if medical disclaimer is needed and present
        medical_terms = ['treatment', 'therapy', 'medication', 'diagnosis', 'symptoms']
        if any(term in output_lower for term in medical_terms):
            if 'educational purposes' not in output_lower and 'medical advice' not in output_lower:
                missing_disclaimers.append("Add medical disclaimer")
        
        # Check for individual variation disclaimer
        if 'individual' not in output_lower and 'vary' not in output_lower:
            if any(term in output_lower for term in ['all women', 'everyone', 'always']):
                missing_disclaimers.append("Add individual variation disclaimer")
        
        return {
            'compliant': len(missing_disclaimers) == 0,
            'missing_disclaimers': missing_disclaimers
        }
    
    def _assess_completeness(self, output: str, context: Dict[str, Any]) -> float:
        """Assess if the output completely addresses the query"""
        # Simplified completeness check
        query = context.get('original_query', '')
        
        if not query:
            return 0.8  # Default score when no query context
        
        # Check if key terms from query are addressed in output
        query_words = set(word_tokenize(query.lower()))
        output_words = set(word_tokenize(output.lower()))
        
        # Remove common stop words
        stop_words = set(stopwords.words('english'))
        query_words -= stop_words
        output_words -= stop_words
        
        if not query_words:
            return 0.8
        
        coverage = len(query_words.intersection(output_words)) / len(query_words)
        return coverage
    
    def _assess_readability(self, output: str) -> float:
        """Assess readability of the output"""
        # Simplified readability assessment
        sentences = sent_tokenize(output)
        words = word_tokenize(output)
        
        if not sentences or not words:
            return 0.0
        
        avg_sentence_length = len(words) / len(sentences)
        avg_word_length = sum(len(word) for word in words) / len(words)
        
        # Simple readability score (lower is better for readability)
        # Inverse relationship: shorter sentences and words = higher score
        readability_score = 100 - (avg_sentence_length * 2) - (avg_word_length * 5)
        
        return max(0.0, min(100.0, readability_score))
    
    def _detect_safety_signals(self, output: str) -> Dict[str, List[str]]:
        """Detect safety signals in output"""
        critical_signals = []
        warning_signals = []
        
        output_lower = output.lower()
        
        # Critical safety signals
        critical_patterns = [
            r'\b(?:guaranteed|definitely|certainly)\s+(?:cure|fix)\b',
            r'\b(?:stop|discontinue)\s+(?:all|your)\s+medication\b',
            r'\b(?:ignore|disregard)\s+(?:doctor|physician)\b'
        ]
        
        for pattern in critical_patterns:
            if re.search(pattern, output_lower):
                critical_signals.append(pattern)
        
        # Warning signals
        warning_patterns = [
            r'\b(?:always|never)\s+(?:works|effective)\b',
            r'\b(?:all|every)\s+(?:women|people)\s+(?:should|must)\b'
        ]
        
        for pattern in warning_patterns:
            if re.search(pattern, output_lower):
                warning_signals.append(pattern)
        
        return {
            'critical_signals': critical_signals,
            'warning_signals': warning_signals
        }

## 7. Audit and Logging System

Comprehensive logging and auditing system for compliance and continuous improvement.

In [ ]:
class AuditLogger:
    """Comprehensive audit logging system"""
    
    def __init__(self, log_level=logging.INFO):
        self.logger = logging.getLogger('safety_validation')
        self.logger.setLevel(log_level)
        
        # Create file handler
        handler = logging.FileHandler('safety_validation.log')
        formatter = logging.Formatter(
            '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        )
        handler.setFormatter(formatter)
        self.logger.addHandler(handler)
        
        # In-memory storage for analytics
        self.audit_records = []
        self.safety_metrics = {
            'total_interactions': 0,
            'blocked_interactions': 0,
            'flagged_interactions': 0,
            'passed_interactions': 0
        }
    
    def log_safety_check(self, result: SafetyCheckResult, interaction_id: str, 
                        user_context: Dict[str, Any] = None):
        """Log safety check results"""
        
        # Create audit record
        audit_record = {
            'interaction_id': interaction_id,
            'timestamp': result.timestamp.isoformat(),
            'check_name': result.check_name,
            'result': result.result.value,
            'safety_level': result.safety_level.value,
            'confidence_score': result.confidence_score,
            'details': result.details,
            'recommendations': result.recommendations,
            'user_context': self._sanitize_user_context(user_context or {})
        }
        
        # Store in memory
        self.audit_records.append(audit_record)
        
        # Log to file
        log_message = f"Safety Check - {result.check_name}: {result.result.value} | "
        log_message += f"Level: {result.safety_level.value} | "
        log_message += f"Confidence: {result.confidence_score:.2f} | "
        log_message += f"Interaction: {interaction_id}"
        
        if result.result == ValidationResult.FAIL:
            self.logger.error(log_message)
        elif result.result == ValidationResult.REVIEW:
            self.logger.warning(log_message)
        else:
            self.logger.info(log_message)
        
        # Update metrics
        self._update_metrics(result)
    
    def log_interaction_summary(self, interaction_id: str, overall_result: ValidationResult,
                              safety_checks: List[SafetyCheckResult]):
        """Log summary of complete interaction validation"""
        
        summary = {
            'interaction_id': interaction_id,
            'timestamp': datetime.datetime.now().isoformat(),
            'overall_result': overall_result.value,
            'total_checks': len(safety_checks),
            'failed_checks': sum(1 for check in safety_checks if check.result == ValidationResult.FAIL),
            'review_checks': sum(1 for check in safety_checks if check.result == ValidationResult.REVIEW),
            'passed_checks': sum(1 for check in safety_checks if check.result == ValidationResult.PASS),
            'avg_confidence': sum(check.confidence_score for check in safety_checks) / len(safety_checks) if safety_checks else 0
        }
        
        self.audit_records.append(summary)
        
        summary_message = f"Interaction Summary - {interaction_id}: {overall_result.value} | "
        summary_message += f"Checks: {summary['total_checks']} | "
        summary_message += f"Failed: {summary['failed_checks']} | "
        summary_message += f"Review: {summary['review_checks']} | "
        summary_message += f"Avg Confidence: {summary['avg_confidence']:.2f}"
        
        self.logger.info(summary_message)
    
    def get_safety_metrics(self) -> Dict[str, Any]:
        """Get current safety metrics"""
        metrics = self.safety_metrics.copy()
        
        if metrics['total_interactions'] > 0:
            metrics['pass_rate'] = metrics['passed_interactions'] / metrics['total_interactions']
            metrics['block_rate'] = metrics['blocked_interactions'] / metrics['total_interactions']
            metrics['flag_rate'] = metrics['flagged_interactions'] / metrics['total_interactions']
        else:
            metrics['pass_rate'] = 0.0
            metrics['block_rate'] = 0.0
            metrics['flag_rate'] = 0.0
        
        return metrics
    
    def generate_safety_report(self) -> Dict[str, Any]:
        """Generate comprehensive safety report"""
        
        report = {
            'report_timestamp': datetime.datetime.now().isoformat(),
            'total_records': len(self.audit_records),
            'metrics': self.get_safety_metrics(),
            'check_performance': self._analyze_check_performance(),
            'trend_analysis': self._analyze_trends(),
            'recommendations': self._generate_recommendations()
        }
        
        return report
    
    def _sanitize_user_context(self, context: Dict[str, Any]) -> Dict[str, Any]:
        """Sanitize user context for logging (remove PII)"""
        sanitized = {}
        
        # Only log non-sensitive context information
        safe_keys = ['age_range', 'interaction_type', 'topic_category', 'timestamp']
        
        for key in safe_keys:
            if key in context:
                sanitized[key] = context[key]
        
        return sanitized
    
    def _update_metrics(self, result: SafetyCheckResult):
        """Update safety metrics based on check result"""
        self.safety_metrics['total_interactions'] += 1
        
        if result.result == ValidationResult.FAIL:
            self.safety_metrics['blocked_interactions'] += 1
        elif result.result == ValidationResult.REVIEW:
            self.safety_metrics['flagged_interactions'] += 1
        else:
            self.safety_metrics['passed_interactions'] += 1
    
    def _analyze_check_performance(self) -> Dict[str, Any]:
        """Analyze performance of different safety checks"""
        check_stats = defaultdict(lambda: {'total': 0, 'failed': 0, 'review': 0, 'passed': 0})
        
        for record in self.audit_records:
            if 'check_name' in record:
                check_name = record['check_name']
                result = record['result']
                
                check_stats[check_name]['total'] += 1
                check_stats[check_name][result] += 1
        
        return dict(check_stats)
    
    def _analyze_trends(self) -> Dict[str, Any]:
        """Analyze safety trends over time"""
        # Simplified trend analysis
        recent_records = [r for r in self.audit_records 
                         if 'timestamp' in r and 
                         datetime.datetime.fromisoformat(r['timestamp']) > 
                         datetime.datetime.now() - datetime.timedelta(days=30)]
        
        return {
            'recent_interactions': len(recent_records),
            'trend_direction': 'stable'  # Simplified
        }
    
    def _generate_recommendations(self) -> List[str]:
        """Generate recommendations based on audit data"""
        recommendations = []
        metrics = self.get_safety_metrics()
        
        if metrics['block_rate'] > 0.1:
            recommendations.append("High block rate detected - review content filtering sensitivity")
        
        if metrics['flag_rate'] > 0.2:
            recommendations.append("High flag rate - consider adjusting review thresholds")
        
        if metrics['pass_rate'] < 0.8:
            recommendations.append("Low pass rate - investigate potential issues in content generation")
        
        return recommendations

## 8. Main Safety Validation Orchestrator

The central component that coordinates all safety validation layers and provides the main interface for the system.

In [ ]:
class SafetyValidationOrchestrator:
    """Main orchestrator for safety validation system"""
    
    def __init__(self):
        # Initialize all safety components
        self.input_validator = InputValidator()
        self.content_filter = ContentSafetyFilter()
        self.medical_validator = MedicalAccuracyValidator()
        self.bias_detector = BiasDetectionSystem()
        self.output_monitor = OutputSafetyMonitor()
        self.audit_logger = AuditLogger()
        
        # Configuration
        self.config = {
            'strict_mode': True,
            'auto_block_threshold': 0.8,
            'review_threshold': 0.6,
            'require_human_review': ['blocked', 'high_risk']
        }
    
    def validate_interaction(self, interaction: UserInteraction) -> Dict[str, Any]:
        """Complete safety validation of user interaction"""
        
        interaction_id = interaction.interaction_id
        safety_checks = []
        
        try:
            # Step 1: Input Validation
            input_result = self.input_validator.validate_input(
                interaction.user_input, 
                interaction.context
            )
            safety_checks.append(input_result)
            self.audit_logger.log_safety_check(input_result, interaction_id, interaction.context)
            
            # Early termination if input is blocked
            if input_result.result == ValidationResult.FAIL:
                return self._create_validation_response(interaction_id, safety_checks, ValidationResult.FAIL)
            
            # Step 2: Content Safety Filtering
            content_result = self.content_filter.filter_content(
                interaction.user_input,
                interaction.context
            )
            safety_checks.append(content_result)
            self.audit_logger.log_safety_check(content_result, interaction_id, interaction.context)
            
            if content_result.result == ValidationResult.FAIL:
                return self._create_validation_response(interaction_id, safety_checks, ValidationResult.FAIL)
            
            # Step 3: Bias Detection
            bias_result = self.bias_detector.detect_bias(
                interaction.user_input,
                interaction.context
            )
            safety_checks.append(bias_result)
            self.audit_logger.log_safety_check(bias_result, interaction_id, interaction.context)
            
            # Determine overall result
            overall_result = self._determine_overall_result(safety_checks)
            
            # Log interaction summary
            self.audit_logger.log_interaction_summary(interaction_id, overall_result, safety_checks)
            
            return self._create_validation_response(interaction_id, safety_checks, overall_result)
            
        except Exception as e:
            # Log error and fail safely
            error_message = f"Safety validation error: {str(e)}"
            self.audit_logger.logger.error(f"Validation error for {interaction_id}: {error_message}")
            
            return {
                'validation_result': ValidationResult.FAIL,
                'safety_level': SafetyLevel.BLOCKED,
                'message': 'Safety validation failed due to system error',
                'allow_processing': False,
                'requires_human_review': True,
                'error': error_message
            }
    
    def validate_output(self, output: str, interaction_id: str, context: Dict[str, Any]) -> Dict[str, Any]:
        """Validate AI-generated output before delivery"""
        
        safety_checks = []
        
        try:
            # Output safety monitoring
            output_result = self.output_monitor.monitor_output(output, context)
            safety_checks.append(output_result)
            self.audit_logger.log_safety_check(output_result, interaction_id, context)
            
            # Medical accuracy validation for output
            medical_result = self.medical_validator.validate_medical_content(output, context)
            safety_checks.append(medical_result)
            self.audit_logger.log_safety_check(medical_result, interaction_id, context)
            
            # Bias detection for output
            bias_result = self.bias_detector.detect_bias(output, context)
            safety_checks.append(bias_result)
            self.audit_logger.log_safety_check(bias_result, interaction_id, context)
            
            # Determine overall result
            overall_result = self._determine_overall_result(safety_checks)
            
            # Log output validation summary
            self.audit_logger.log_interaction_summary(f"{interaction_id}_output", overall_result, safety_checks)
            
            response = self._create_validation_response(interaction_id, safety_checks, overall_result)
            
            # Add output-specific information
            response['validated_output'] = output if overall_result != ValidationResult.FAIL else None
            response['output_modifications'] = self._generate_output_modifications(safety_checks)
            
            return response
            
        except Exception as e:
            error_message = f"Output validation error: {str(e)}"
            self.audit_logger.logger.error(f"Output validation error for {interaction_id}: {error_message}")
            
            return {
                'validation_result': ValidationResult.FAIL,
                'safety_level': SafetyLevel.BLOCKED,
                'message': 'Output validation failed due to system error',
                'allow_delivery': False,
                'requires_human_review': True,
                'error': error_message
            }
    
    def get_system_status(self) -> Dict[str, Any]:
        """Get current system status and metrics"""
        
        return {
            'system_status': 'operational',
            'safety_metrics': self.audit_logger.get_safety_metrics(),
            'configuration': self.config,
            'component_status': {
                'input_validator': 'active',
                'content_filter': 'active',
                'medical_validator': 'active',
                'bias_detector': 'active',
                'output_monitor': 'active',
                'audit_logger': 'active'
            },
            'last_updated': datetime.datetime.now().isoformat()
        }
    
    def generate_safety_report(self) -> Dict[str, Any]:
        """Generate comprehensive safety report"""
        return self.audit_logger.generate_safety_report()
    
    def update_configuration(self, new_config: Dict[str, Any]):
        """Update system configuration"""
        self.config.update(new_config)
        self.audit_logger.logger.info(f"Configuration updated: {new_config}")
    
    def _determine_overall_result(self, safety_checks: List[SafetyCheckResult]) -> ValidationResult:
        """Determine overall validation result from individual checks"""
        
        # If any check failed, overall result is fail
        if any(check.result == ValidationResult.FAIL for check in safety_checks):
            return ValidationResult.FAIL
        
        # If any check requires review, overall result is review
        if any(check.result == ValidationResult.REVIEW for check in safety_checks):
            return ValidationResult.REVIEW
        
        # All checks passed
        return ValidationResult.PASS
    
    def _create_validation_response(self, interaction_id: str, safety_checks: List[SafetyCheckResult], 
                                  overall_result: ValidationResult) -> Dict[str, Any]:
        """Create standardized validation response"""
        
        # Determine highest safety level
        safety_levels = [check.safety_level for check in safety_checks]
        highest_safety_level = max(safety_levels, key=lambda x: ['safe', 'caution', 'warning', 'blocked'].index(x.value))
        
        # Collect all recommendations
        all_recommendations = []
        for check in safety_checks:
            all_recommendations.extend(check.recommendations)
        
        # Calculate average confidence
        avg_confidence = sum(check.confidence_score for check in safety_checks) / len(safety_checks) if safety_checks else 0
        
        # Determine if human review is required
        requires_human_review = (
            overall_result == ValidationResult.FAIL or
            highest_safety_level == SafetyLevel.BLOCKED or
            avg_confidence < self.config['review_threshold']
        )
        
        return {
            'interaction_id': interaction_id,
            'validation_result': overall_result,
            'safety_level': highest_safety_level,
            'confidence_score': avg_confidence,
            'allow_processing': overall_result != ValidationResult.FAIL,
            'requires_human_review': requires_human_review,
            'safety_checks': [{
                'check_name': check.check_name,
                'result': check.result.value,
                'safety_level': check.safety_level.value,
                'confidence': check.confidence_score
            } for check in safety_checks],
            'recommendations': list(set(all_recommendations)),  # Remove duplicates
            'timestamp': datetime.datetime.now().isoformat()
        }
    
    def _generate_output_modifications(self, safety_checks: List[SafetyCheckResult]) -> List[str]:
        """Generate suggested output modifications based on safety checks"""
        modifications = []
        
        for check in safety_checks:
            if check.check_name == 'output_safety_monitor':
                if 'disclaimer_compliance' in check.details:
                    compliance = check.details['disclaimer_compliance']
                    if not compliance.get('compliant', True):
                        modifications.extend(compliance.get('missing_disclaimers', []))
            
            elif check.check_name == 'bias_detection':
                if check.safety_level in [SafetyLevel.WARNING, SafetyLevel.CAUTION]:
                    modifications.append("Review for inclusive language")
            
            elif check.check_name == 'medical_accuracy_validation':
                if 'red_flags' in check.details and check.details['red_flags']:
                    modifications.append("Soften definitive medical statements")
        
        return list(set(modifications))  # Remove duplicates

## 9. Demonstration and Testing

Example usage and testing of the complete safety validation system.

In [ ]:
def demonstrate_safety_system():
    """Demonstrate the safety validation system with various test cases"""
    
    # Initialize the safety validation system
    safety_system = SafetyValidationOrchestrator()
    
    print("=== Menopause Education AI Safety Validation System Demo ===")
    print()
    
    # Test cases
    test_cases = [
        {
            'name': 'Safe Educational Query',
            'input': 'What are the common symptoms of menopause and how long do they typically last?',
            'context': {'age_range': '45-55', 'topic_category': 'symptoms'}
        },
        {
            'name': 'Medical Advice Seeking (Should be flagged)',
            'input': 'Should I stop taking my hormone replacement therapy? I want a diagnosis.',
            'context': {'age_range': '50-60', 'topic_category': 'treatment'}
        },
        {
            'name': 'Potentially Biased Content',
            'input': 'Do all women experience the same menopause symptoms?',
            'context': {'age_range': '40-50', 'topic_category': 'general'}
        },
        {
            'name': 'Off-topic Query (Should be flagged)',
            'input': 'How do I invest in cryptocurrency?',
            'context': {'age_range': '30-40', 'topic_category': 'finance'}
        },
        {
            'name': 'Emotional Distress Indicator',
            'input': 'I feel terrible and depressed about these menopause changes. Nothing helps.',
            'context': {'age_range': '48-52', 'topic_category': 'emotional_support'}
        }
    ]
    
    # Test input validation
    print("1. Testing Input Validation:")
    print("="*50)
    
    for i, test_case in enumerate(test_cases, 1):
        print(f"\nTest Case {i}: {test_case['name']}")
        print(f"Input: {test_case['input']}")
        
        # Create user interaction
        interaction = UserInteraction(
            session_id=f"session_{i}",
            user_input=test_case['input'],
            user_demographics={},
            context=test_case['context'],
            timestamp=datetime.datetime.now(),
            interaction_id=f"interaction_{i}"
        )
        
        # Validate interaction
        result = safety_system.validate_interaction(interaction)
        
        print(f"Result: {result['validation_result'].value}")
        print(f"Safety Level: {result['safety_level'].value}")
        print(f"Confidence: {result['confidence_score']:.2f}")
        print(f"Allow Processing: {result['allow_processing']}")
        print(f"Requires Review: {result['requires_human_review']}")
        
        if result['recommendations']:
            print(f"Recommendations: {', '.join(result['recommendations'])}")
        
        print("-" * 30)
    
    # Test output validation
    print("\n\n2. Testing Output Validation:")
    print("="*50)
    
    sample_outputs = [
        {
            'name': 'Good Educational Response',
            'output': '''Menopause typically occurs between ages 45-55, with the average age being 51. Common symptoms include hot flashes, night sweats, irregular periods, and mood changes. These symptoms can vary significantly between individuals. 
            
            This information is for educational purposes only and should not replace professional medical advice. Individual experiences may vary, so please consult with a healthcare provider for personalized guidance.''',
            'context': {'original_query': 'What are menopause symptoms?'}
        },
        {
            'name': 'Problematic Medical Advice',
            'output': '''You should definitely stop taking hormone therapy immediately. All women experience the same side effects and this treatment never works for anyone over 50.''',
            'context': {'original_query': 'Is hormone therapy safe?'}
        }
    ]
    
    for i, output_test in enumerate(sample_outputs, 1):
        print(f"\nOutput Test {i}: {output_test['name']}")
        print(f"Output: {output_test['output'][:100]}...")
        
        # Validate output
        output_result = safety_system.validate_output(
            output_test['output'],
            f"output_test_{i}",
            output_test['context']
        )
        
        print(f"Result: {output_result['validation_result'].value}")
        print(f"Safety Level: {output_result['safety_level'].value}")
        print(f"Allow Delivery: {output_result.get('allow_delivery', 'N/A')}")
        
        if output_result.get('output_modifications'):
            print(f"Suggested Modifications: {', '.join(output_result['output_modifications'])}")
        
        print("-" * 30)
    
    # Display system status
    print("\n\n3. System Status:")
    print("="*50)
    
    status = safety_system.get_system_status()
    print(f"System Status: {status['system_status']}")
    print(f"Total Interactions: {status['safety_metrics']['total_interactions']}")
    print(f"Pass Rate: {status['safety_metrics']['pass_rate']:.1%}")
    print(f"Block Rate: {status['safety_metrics']['block_rate']:.1%}")
    print(f"Flag Rate: {status['safety_metrics']['flag_rate']:.1%}")
    
    # Generate safety report
    print("\n\n4. Safety Report:")
    print("="*50)
    
    report = safety_system.generate_safety_report()
    print(f"Report Generated: {report['report_timestamp']}")
    print(f"Total Audit Records: {report['total_records']}")
    
    if report['recommendations']:
        print("\nSystem Recommendations:")
        for rec in report['recommendations']:
            print(f"- {rec}")
    
    print("\n=== Demo Complete ===")
    return safety_system

# Run the demonstration
if __name__ == "__main__":
    safety_system = demonstrate_safety_system()

## 10. Configuration and Customization

Additional configuration options and customization capabilities for different deployment scenarios.

In [ ]:
# Configuration examples and testing utilities

def create_production_config():
    """Create production-ready configuration"""
    return {
        'strict_mode': True,
        'auto_block_threshold': 0.9,
        'review_threshold': 0.7,
        'require_human_review': ['blocked', 'high_risk', 'medical_advice'],
        'logging_level': 'INFO',
        'audit_retention_days': 365,
        'enable_real_time_monitoring': True,
        'compliance_mode': 'HIPAA',
        'cultural_sensitivity_level': 'high'
    }

def create_development_config():
    """Create development configuration for testing"""
    return {
        'strict_mode': False,
        'auto_block_threshold': 0.8,
        'review_threshold': 0.5,
        'require_human_review': ['blocked'],
        'logging_level': 'DEBUG',
        'audit_retention_days': 30,
        'enable_real_time_monitoring': False,
        'compliance_mode': 'development',
        'cultural_sensitivity_level': 'medium'
    }

def run_safety_test_suite(safety_system: SafetyValidationOrchestrator):
    """Run comprehensive test suite for safety validation system"""
    
    print("Running Safety Validation Test Suite...")
    print("="*60)
    
    test_results = {
        'passed': 0,
        'failed': 0,
        'total': 0
    }
    
    # Test cases with expected results
    test_cases = [
        {
            'name': 'Safe Query Test',
            'input': 'What are common menopause symptoms?',
            'expected_result': ValidationResult.PASS,
            'expected_safety_level': SafetyLevel.SAFE
        },
        {
            'name': 'Medical Advice Block Test',
            'input': 'Should I stop my medication and diagnose myself?',
            'expected_result': ValidationResult.FAIL,
            'expected_safety_level': SafetyLevel.BLOCKED
        },
        {
            'name': 'Off-topic Review Test',
            'input': 'How do I buy stocks?',
            'expected_result': ValidationResult.REVIEW,
            'expected_safety_level': SafetyLevel.CAUTION
        }
    ]
    
    for i, test_case in enumerate(test_cases, 1):
        test_results['total'] += 1
        
        # Create test interaction
        interaction = UserInteraction(
            session_id=f"test_session_{i}",
            user_input=test_case['input'],
            user_demographics={},
            context={'test': True},
            timestamp=datetime.datetime.now(),
            interaction_id=f"test_{i}"
        )
        
        # Run validation
        result = safety_system.validate_interaction(interaction)
        
        # Check results
        passed = (
            result['validation_result'] == test_case['expected_result'] and
            result['safety_level'] == test_case['expected_safety_level']
        )
        
        if passed:
            test_results['passed'] += 1
            status = "✓ PASS"
        else:
            test_results['failed'] += 1
            status = "✗ FAIL"
        
        print(f"{status} - {test_case['name']}")
        print(f"    Expected: {test_case['expected_result'].value}/{test_case['expected_safety_level'].value}")
        print(f"    Actual:   {result['validation_result'].value}/{result['safety_level'].value}")
        print()
    
    # Print summary
    print("Test Summary:")
    print(f"Total Tests: {test_results['total']}")
    print(f"Passed: {test_results['passed']}")
    print(f"Failed: {test_results['failed']}")
    print(f"Success Rate: {test_results['passed']/test_results['total']:.1%}")
    
    return test_results

# Example usage
print("\n=== Configuration Examples ===")
prod_config = create_production_config()
dev_config = create_development_config()

print("Production Config:")
for key, value in prod_config.items():
    print(f"  {key}: {value}")

print("\nDevelopment Config:")
for key, value in dev_config.items():
    print(f"  {key}: {value}")

# Run test suite if safety_system exists
if 'safety_system' in locals():
    print("\n=== Running Test Suite ===")
    test_results = run_safety_test_suite(safety_system)
else:
    print("\nTo run the test suite, first execute the demonstration section above.")

## Conclusion

This comprehensive safety validation system provides multiple layers of protection for a menopause education AI system:

### Key Features:

1. **Multi-layered Validation**: Input validation, content filtering, medical accuracy checking, bias detection, and output monitoring
2. **Comprehensive Logging**: Full audit trail for compliance and continuous improvement
3. **Configurable Thresholds**: Adaptable to different deployment scenarios
4. **Real-time Monitoring**: Continuous safety assessment during operation
5. **Human Review Integration**: Automatic flagging for human review when needed

### Safety Standards Addressed:

- **Medical Safety**: Prevents harmful medical advice and ensures appropriate disclaimers
- **Bias Mitigation**: Detects and addresses cultural, age, and socioeconomic biases
- **Content Safety**: Filters inappropriate or off-topic content
- **Compliance**: Built with HIPAA and medical device standards in mind
- **Accessibility**: Considers diverse user needs and abilities

### Usage:

1. Initialize the `SafetyValidationOrchestrator`
2. Use `validate_interaction()` for incoming user queries
3. Use `validate_output()` for AI-generated responses
4. Monitor system health with `get_system_status()`
5. Generate compliance reports with `generate_safety_report()`

This system ensures that the menopause education AI operates safely, ethically, and in compliance with healthcare standards while providing valuable educational support to users.